[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyneuro/4590_Colabs/blob/main/5_SpikAdaptBurst.ipynb)

# Set 5 — From Passive → Spiking, Adaptation & Bursting
**Author:**Neural Engineering Laboratory, University of Missouri

Gregory Glickert, Khuram Choudhry, Ziao Chen, Satish S. Nair

---
### Module Overview
This module is framed as a **design challenge**: you have a passive neuron (bilayer + leak channels) and need to engineer it into a spiking one, then design variants with different output profiles. You will:
1. **Design a spiker:** find the minimum current (Rheobase) to fire a spike at all.
2. **Design for refractoriness:** discover why a neuron can't fire twice instantly.
3. **Design an adapter:** make a cell slow its firing rate over time.
4. **Design a burster:** create complex firing patterns using slow envelopes.

An **optional deep-dive** at the end covers the gate-level mechanics (m, h, n) for anyone curious — skippable without losing anything from the main module.

---

<details>
<summary><b>📖 Click to expand: Module Reference & Module Roadmap</b></summary>

### Variable Key: Set 5 Parameters
| Symbol | NEURON Name | Definition | Units |
| :--- | :--- | :--- | :--- |
| **$g_{pas}$** | `g_pas` | Leak conductance (Passive membrane "holeyness") | $S/cm^2$ |
| **$c_m$** | `cm` | Membrane Capacitance (The "battery" effect) | $\mu F/cm^2$ |
| **$g_{Na}$** | `gnabar_hh` | Maximum Sodium Conductance (The Upstroke) | $S/cm^2$ |
| **$g_{K}$** | `gkbar_hh` | Maximum Potassium Conductance (The Downstroke) | $S/cm^2$ |
| **$R_a$** | `ra` | Axial Resistance (Fluid resistance inside the cable) | $\Omega \cdot cm$ |
| **$g_M$** | `gm` | M-current Conductance (Slow "brake" for adaptation) | $S/cm^2$ |

---

### Glossary
* **Rheobase:** The minimum current amplitude required to elicit a spike given a long-duration pulse.
* **Refractory Period:** The recovery time post-spike. **Absolute** vs. **Relative**.
* **$dV/dt$:** The first derivative of voltage; represents the "speed" of the spike upstroke.
* **Adaptation:** The gradual increase in Inter-Spike Intervals (ISI) during a constant stimulus.
* **Phase-Plane:** A plot of $V$ vs. $dV/dt$ used to visualize the "trajectory" of a spike.
* **ISI (Inter-Spike Interval):** The time duration between the peaks of two consecutive action potentials.

---

### Module Roadmap
* **The Design Challenge** — thought experiment + AI-discovery of voltage-gated channels
* **D0** — Passive Recap: The Baseline
* **D1** — Add HH & Threshold Search
* **D2** — Ionic Logic: Na⁺ vs. K⁺ (+ by-hand equilibrium drill)
* **D3** — Refractory (Paired Pulses)
* **D4** — f–I Curve & Rheobase (+ optional strength-duration extension)
* **D5** — Waveform Landmarks
* **D6** — Propagation Teaser
* **D7** — Design Challenge: Build an Adapting Neuron
* **D8** — Quantify Adaptation
* **D9** — Design Challenge: Build a Bursting Neuron
* **D10** — AHP/ADP Notes
* **D11** — Phase-Plane Glimpse
* **D12** — Numerical Tips
* **Homework + Knowledge Bank**
* **OPTIONAL: Gate Mechanics Deep Dive** — skippable

</details>

---
### A Note on This Notebook
**This completed, filled-in notebook is the lab deliverable.** Add your own cells as you go. **D11 — Phase-plane glimpse** below is a good spot for an optional (AI-assisted) extension.

---
### 📋 Before You Begin: How This Module Is Graded
This module's submission and its companion quiz are weighted **equally**. The quiz specifically checks whether the understanding behind your submission is actually yours — pasting a friend's slider values or an AI's explanation can get this notebook looking complete, but it won't get you through a quiz question applying the same concept to a scenario you haven't seen before.

The **design problem** below, and the **two questions you'll create yourself**, both exist for exactly this reason.

---

### Starter / Initialization

**Purpose**
Load the necessary Hodgkin-Huxley (HH) mechanisms, plotting tools, and specialized NEURON libraries. This step confirms the environment is properly configured for biophysical simulations.

---

### Post-Run Checklist
After executing the initialization code, record the following in your notes:
*   **NEURON Version:** Note the specific version string printed during import.
*   **Import Messages:** Document any specific warnings or "success" messages that appear.
*   **Initial Defaults:** List the starting values for $R_a$, $C_m$, the leak conductance, and the standard HH channel parameters (e.g., $g_{Na}$, $g_K$).

---

### Exercises

**Clamp Functionality**
State in one concise line: What does a current clamp specifically control, and what does it measure?
*   [Your Answer Here]

**Figure Requirements**
List three essential items that must appear in every figure generated in this notebook (e.g., units on axes, legend/labels, protocol parameters).
1.  
2.  
3.  

**Seed Control and Reproducibility**
Explain, in one or two sentences, why manual seed control is not required for these specific HH simulations but why it will become critical once noise or stochastic channel openings are introduced.
*   [Your Answer Here]

## The Design Challenge: Building a Spiking Neuron
So far, every neuron you've built (Set 3's Module A/B, the passive rigs) can respond to current — but none of them can **spike**. Before diving into the rigs below, let's think like an engineer, not just a user of pre-built tools.

**🚫 Thought Experiment — No AI:**
1. Sketch what an action potential looks like: membrane voltage over time, rising sharply from rest (~-65 mV) to a peak (~+20 to +40 mV), then falling back down. Label the axes.
2. Here's the real challenge: **you have nothing but a lipid bilayer and some leak channels** (Set 3's passive neuron). How would you make this membrane produce that sharp voltage spike? List as many different ways as you can think of to make $V_m$ rise quickly — don't worry about biological accuracy yet, just brainstorm mechanisms.

*Your Answer Here (sketch description + your brainstormed list):*

**Activity — Ask AI:**

> *"I have a neuron with only a lipid bilayer (capacitor) and leak channels (fixed resistors). What would I need to add to make this membrane produce a sudden, large voltage spike in response to a stimulus? Explain the general idea, not the exact biology yet."*

Paste the response, then compare it to your own brainstormed list above — did AI suggest anything you didn't think of, or vice versa?

*Your Answer Here:*

**The answer biology settled on:** channels whose resistance *changes* with voltage — **voltage-gated channels**. Instead of a fixed resistor, picture a resistor that gets dramatically less resistive (opens) once voltage crosses some level. This is the one new idea this entire module builds on: **a channel that acts as a variable resistor, controlled by voltage itself.**

We are *not* going to open up exactly how that gating works yet (that's an optional deep-dive at the end, for anyone curious) — for now, the goal is to use this new kind of component to **design neurons with different output profiles**: ones that fire once, ones that fire steadily, ones that adapt, ones that burst. Every section below is really a design challenge in disguise.

In [ ]:
#@title Starter / Initialization { display-mode: "form" }
#@markdown Run this cell to initialize the NEURON environment and load HH mechanisms.

import numpy as np
import matplotlib.pyplot as plt
from neuron import h
from neuron.units import ms, mV

# Standard NEURON setup
h.load_file('stdrun.hoc')
h.dt = 0.025 # Standard integration step
h.steps_per_ms = 1/h.dt

# Configure plot styling for the whole notebook
plt.rcParams.update({
    'font.size': 12,
    'lines.linewidth': 2,
    'axes.spines.top': False,
    'axes.spines.right': False
})

print("Set 5 Environment: Ready.")

## <a id="D0"></a>D0 — Passive Recap: The Baseline

Before we add "sparks" (spikes), we must ensure our passive baseline is correct.
* **Goal:** Estimate the time constant $\tau$ and Input Resistance $R_{in}$.
* **Expectation:** $\tau \approx R_{in} \cdot C_m$.

---

### What to Change
* **Step Amplitude/Duration:** Adjust the stimulus parameters to see different deflection levels.
* **Passive Properties:** Modify leak conductance ($g_{pas}$) and capacitance ($c_m$) using the sliders.

### Model Verification
* **Time Constant:** Mark $\tau$ at 63.2% of the steady-state voltage step.
* **Input Resistance:** Compute $R_{in} = \Delta V / \Delta I$ at steady state.
* **Consistency:** Verify that $\tau \approx R_{in} \cdot C_m$ (Check your units!).

### Predict → Verify
* **Capacitance:** $\uparrow c_m \rightarrow \uparrow \tau$.
* **Conductance:** $\uparrow g_{pas} \rightarrow \downarrow R_{in}$ and $\downarrow \tau$.

---

### Exercises

**1. Capacitance Scaling** Double $c_m$ and re-measure $\tau$. Report the new value and how it compares to the baseline.

**2. Conductance Scaling** Halve $g_{pas}$ and report both the new $R_{in}$ and $\tau$.

**3. Conceptual Logic** In one sentence: Why is it important to verify passive membrane baselines before introducing active spiking conductances?
> *Your Answer Here:*

In [ ]:
#@title D0: Interactive Passive Explorer { display-mode: "form" }
#@markdown Adjust the sliders and the plot will update automatically.

import numpy as np
import matplotlib.pyplot as plt
from neuron import h
from neuron.units import ms, mV

# --- Parameters ---
g_pas = 0.0002 #@param {type:"slider", min:0.00001, max:0.001, step:0.00001}
cm = 1.0 #@param {type:"slider", min:0.1, max:5.0, step:0.1}
stim_amp = -0.05 #@param {type:"slider", min:-0.2, max:0.2, step:0.01}
stim_dur = 40 #@param {type:"slider", min:10, max:200, step:10}

def run_simulation(g, c, amp, dur):
    h('forall delete_section()')
    soma = h.Section(name='soma_recap')
    soma.L = soma.diam = 20
    soma.insert('pas')
    soma.g_pas = g
    soma.cm = c
    soma.e_pas = -65

    stim = h.IClamp(soma(0.5))
    stim.delay = 5
    stim.dur = dur
    stim.amp = amp

    v = h.Vector().record(soma(0.5)._ref_v)
    t = h.Vector().record(h._ref_t)

    h.finitialize(-65 * mV)
    h.continuerun((dur + 20) * ms)

    plt.figure(figsize=(8, 4))
    plt.plot(t, v, color='black', label=f'Current Step: {amp} nA')
    plt.axhline(-65, linestyle='--', color='gray', alpha=0.5, label='Rest (-65mV)')

    v_steady = v.as_numpy()[-1]
    v_diff = v_steady - (-65)
    v_tau = -65 + (0.632 * v_diff)
    plt.axhline(v_tau, linestyle=':', color='red', alpha=0.6, label='63% (Tau Target)')

    plt.title(f"D0: Passive Response (g_pas={g}, cm={c})")
    plt.xlabel("Time (ms)")
    plt.ylabel("Voltage (mV)")
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    plt.show()

run_simulation(g_pas, cm, stim_amp, stim_dur)

<div id="D1"></div>
## D1 — Add HH & Threshold Search

**Design framing:** you've just added voltage-gated channels to your passive neuron. First design question: **what's the minimum "push" needed to make it spike at all?**

**Idea:** Move from passive to spiking dynamics. The goal is to determine the **rheobase** (the minimum current required to elicit a spike) and examine how the **first-spike latency** changes as a function of the injected current.

---

### What to Change
* **Step Amplitudes:** Use the slider to sweep through current values near the suspected threshold to pinpoint the rheobase.
* **Duration:** Ensure the pulse duration is sufficient for a spike to occur, especially at near-threshold currents where latency is high.

### Model Validation
* **Spike Detection:** For each current amplitude, report "spike" or "no spike."
* **Latency Recording:** For every spike, record the **first-spike latency** (the time from stimulus onset to the peak of the first action potential).
* **Visual Verification:** Observe the sharp decrease in latency just above rheobase.

### Predict → Verify
* **Latency:** Does latency decrease linearly or nonlinearly as you increase current?
* **The "Kink":** Notice how, at currents just above threshold, the spike takes a long time to "climb" out of the baseline.

---

### Exercises

**1. Threshold Table** Build a small summary table in your notes with the following columns:
* Injected Current (I)
* Spike? (Yes/No)
* First-spike Latency (ms)

**2. Rheobase Analysis** Identify the smallest current ($I$) that successfully elicits a spike.
* **Explain:** Discuss how uncertainty or "jitter" arises when trying to determine a precise threshold near the rheobase in a computational or biological system.

**3. Qualitative Sketching** Describe the shape of the curve for the first-spike latency versus the injected current for values just above the threshold.
> *Your Answer Here:*

In [ ]:
#@title D1: Interactive Threshold Explorer { display-mode: "form" }
#@markdown Adjust the current to find the exact Rheobase (the point where the first spike appears).

import numpy as np
import matplotlib.pyplot as plt
from neuron import h
from neuron.units import ms, mV

# --- Parameters ---
stim_amp = 0.05 #@param {type:"slider", min:0.0, max:0.2, step:0.001}
stim_dur = 20 #@param {type:"slider", min:5, max:100, step:5}

def run_threshold_sim(amp, dur):
    h('forall delete_section()')
    soma = h.Section(name='soma_hh')
    soma.L = soma.diam = 20
    soma.insert('pas')
    soma.insert('hh') # Standard Hodgkin-Huxley

    stim = h.IClamp(soma(0.5))
    stim.delay = 5
    stim.dur = dur
    stim.amp = amp

    v = h.Vector().record(soma(0.5)._ref_v)
    t = h.Vector().record(h._ref_t)

    h.finitialize(-65 * mV)
    h.continuerun((dur + 15) * ms)

    # Calculate Latency
    v_np = v.as_numpy()
    t_np = t.as_numpy()
    peak_idx = np.argmax(v_np)
    peak_val = v_np[peak_idx]

    has_spike = "YES" if peak_val > 0 else "NO"
    latency = t_np[peak_idx] - 5 if has_spike == "YES" else 0

    # Plotting
    plt.figure(figsize=(9, 4))
    plt.plot(t, v, color='blue', label=f'Current: {amp} nA')
    plt.axhline(0, linestyle='--', color='red', alpha=0.3, label='Spike Threshold (0mV)')

    title_str = f"D1: Threshold Search | Spike: {has_spike}"
    if has_spike == "YES":
        title_str += f" | Latency: {latency:.2f} ms"
        plt.plot(t_np[peak_idx], peak_val, 'ro') # Mark the peak

    plt.title(title_str)
    plt.xlabel("Time (ms)")
    plt.ylabel("Voltage (mV)")
    plt.ylim(-80, 50)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

run_threshold_sim(stim_amp, stim_dur)

<div id="D2"></div>
## D2 — Ionic logic: Na⁺ upstroke vs. K⁺ downstroke

**Idea:** The shape of an action potential is determined by the kinetic interplay between different ion channels. We can attribute specific spike phases to the **Na⁺ upstroke** and the **K⁺-dominated downstroke/AHP** by selectively scaling the maximum conductances ($g_{Na}$ or $g_{K}$).

---

### What to Change
* **Sodium Conductance Scale:** Adjust the multiplier for `gnabar_hh`. Setting this to 0.7 simulates a 30% reduction.
* **Potassium Conductance Scale:** Adjust the multiplier for `gkbar_hh`.
* **Current Injection:** Ensure the current is high enough to trigger a spike even when conductances are reduced.

### Validation Steps
* **Reduced $g_{Na}$:** Observe if a higher current is required to reach threshold and if the spike upstroke (depolarization phase) becomes visibly slower and the peak becomes lower.
* **Reduced $g_{K}$:** Observe if the repolarization phase (downstroke) slows down and if the action potential width increases.
* **Phase Analysis:** Look at the $dV/dt$ vs $V$ plot (if enabled) or the slope of the upstroke to see the "velocity" of the spike.

**Activity — Ask AI:** *"In a spiking neuron, why does sodium entering the cell cause the sharp upstroke, while potassium leaving the cell causes the downstroke? Explain in terms of which direction each ion's concentration gradient pushes it."* Paste the response.

*Your Answer Here:*

### Predict → Verify
* **Control vs. Low $g_{Na}$:** How does sodium availability affect the "all-or-none" nature of the spike?
* **Control vs. Low $g_{K}$:** Without sufficient potassium, does the cell return to rest as quickly?

---

### Exercises

**1. Rheobase Sensitivity** Reduce the maximum sodium conductance ($g_{Na}$) by **30%** (set scale to 0.7).
* **Report:** What is the new rheobase compared to your baseline in D1?

**2. Action Potential Width** Reduce the maximum potassium conductance ($g_{K}$) by **30%** (set scale to 0.7).
* **Measure:** Quantify the change in the action potential **half-width** (the duration of the spike at 50% of its peak amplitude).

**3. Mechanistic Conclusion** In two sentences, explain how these specific conductance manipulations clarify the individual roles of Na⁺ and K⁺ ions in generating the action potential waveform.
> *Your Answer Here:*

In [ ]:
#@title D2: Ionic Conductance Toggle { display-mode: "form" }
#@markdown Adjust scales to see how Na and K shape the spike. (1.0 = Default HH)

gnabar_scale = 1.0 #@param {type:"slider", min:0.0, max:2.0, step:0.1}
gkbar_scale = 1.0 #@param {type:"slider", min:0.0, max:2.0, step:0.1}
stim_amp = 0.1 #@param {type:"slider", min:0.0, max:0.5, step:0.01}

def run_ionic_logic(gna_s, gk_s, amp):
    h('forall delete_section()')
    soma = h.Section(name='soma_ionic')
    soma.L = soma.diam = 20
    soma.insert('hh')

    # Apply scales to standard HH values
    # Default gnabar: 0.12, gkbar: 0.036
    soma.gnabar_hh = 0.12 * gna_s
    soma.gkbar_hh = 0.036 * gk_s

    stim = h.IClamp(soma(0.5))
    stim.delay = 5
    stim.dur = 20
    stim.amp = amp

    v = h.Vector().record(soma(0.5)._ref_v)
    t = h.Vector().record(h._ref_t)

    # Run Simulation
    h.finitialize(-65)
    h.continuerun(40)

    # Plotting
    plt.figure(figsize=(10, 5))

    # Plot Modified Trace
    plt.plot(t, v, 'b-', linewidth=2, label=f'Modified (Na:{gna_s}x, K:{gk_s}x)')

    # Logic for a "Control" reference (Standard HH)
    plt.axhline(-65, color='k', linestyle=':', alpha=0.3)
    plt.title("D2: Impact of Ion Channel Conductance on Spike Shape")
    plt.xlabel("Time (ms)")
    plt.ylabel("Membrane Potential (mV)")
    plt.ylim(-80, 50)
    plt.grid(True, alpha=0.2)
    plt.legend()
    plt.show()

run_ionic_logic(gnabar_scale, gkbar_scale, stim_amp)

### 🚫 By-Hand Drill — No AI, No Code
Every exercise so far has you *run* a simulation and *observe*. This one is different: verify, by hand, that the model is internally consistent at equilibrium.

At rest (before any current injection), the membrane isn't doing nothing — currents through Na⁺, K⁺, and leak channels are all flowing, but they must **sum to exactly zero** (otherwise $V_m$ wouldn't be at a stable rest).

Given parameters: $\bar{g}_{Na}=0.12$, $\bar{g}_{Kd}=0.036$, $\bar{g}_{leak}=0.0003$ S/cm², $A_{soma}=0.00785$ cm², $C_{soma}=1$ μF/cm².

**Steps:**
1. In your Colab, access the resting values from NEURON: run `access soma` then read off `v`, `ena`, `ek`, `el_leak` (or your model's leak reversal), and the gating variables `m_hh`, `h_hh`, `n_hh` at rest (with $I_{inject}=0$).
2. **By hand**, compute each current using $I_x = A_{soma}\cdot\bar{g}_x \cdot (\text{gating term}) \cdot (V_{rest}-E_x)$, where the gating term is $m^3h$ for Na⁺ and $n^4$ for K⁺ (leak has no gating term).
3. Show that $I_{Na} + I_K + I_{leak} \approx 0$ (small rounding error is fine).

*Your Answer Here (show all three current calculations and their sum):*

## D3 — Refractory (paired pulses)

**Idea:** Separate **absolute** vs. **relative** refractoriness by using two identical pulses at varying Inter-Stimulus Intervals (ISIs). This reveals the time needed for ion channels (particularly Na⁺) to recover from inactivation.

---

### What to Change
* **Inter-Stimulus Interval (ISI):** Use the slider to move the second pulse closer to or further from the first.
* **Pulse Amplitude:** Set this to a "supra-threshold" value (slightly above the rheobase found in D1).

### Validation Steps
* **Absolute Refractory Period:** Identify the ISI range where the second pulse *never* elicits a spike, no matter the amplitude.
* **Relative Refractory Period:** Identify the ISI range where a second spike is possible, but may require more current or show a reduced peak.
* **Boundary Definition:** Note the exact ISI where the second spike first reappears.

### Predict → Verify
* **Recovery:** As the ISI increases, does the height of the second spike increase or stay the same?
* **Na+ Inactivation:** Relate the "failure" of the second spike to the percentage of sodium channels still in the inactivated state.

---

### Exercises

**1. Refractory Mapping** Using the interactive tool, find the "Success Boundary."
* **Mark:** What is the duration (in ms) of the Absolute Refractory Period for this cell?

**2. Excitability Recovery** Choose a fixed ISI where the second spike initially fails. Gradually increase the amplitude of the second pulse (if the slider allows or by note) until spiking returns.
* **Report:** Note the required current increment compared to the first pulse to "force" a spike during the relative refractory period.

**3. Rate Limiting** In two sentences, explain how the existence of a refractory period sets a fundamental physical limit on a neuron's maximum firing rate.
> *Your Answer Here:*

In [ ]:
#@title D3: Paired-Pulse Recovery Explorer { display-mode: "form" }
#@markdown Adjust the Interval (ISI) to see when the neuron is ready to fire again.

isi = 8 #@param {type:"slider", min:1, max:30, step:0.5}
pulse_amp = 0.12 #@param {type:"slider", min:0.05, max:0.5, step:0.01}
pulse_dur = 2 #@param {type:"slider", min:1, max:10, step:0.5}

def run_refractory_sim(interval, amp, dur):
    h('forall delete_section()')
    soma = h.Section(name='soma_refractory')
    soma.L = soma.diam = 20
    soma.insert('hh')

    # Pulse 1
    stim1 = h.IClamp(soma(0.5))
    stim1.delay = 10
    stim1.dur = dur
    stim1.amp = amp

    # Pulse 2
    stim2 = h.IClamp(soma(0.5))
    stim2.delay = 10 + dur + interval
    stim2.dur = dur
    stim2.amp = amp

    v = h.Vector().record(soma(0.5)._ref_v)
    t = h.Vector().record(h._ref_t)

    h.finitialize(-65)
    h.continuerun(10 + dur + interval + 20)

    # Plotting
    plt.figure(figsize=(10, 4))
    plt.plot(t, v, color='darkgreen', label=f'ISI: {interval} ms')
    plt.fill_between([10, 10+dur], -80, 40, color='blue', alpha=0.1, label='Pulse 1')
    plt.fill_between([10+dur+interval, 10+2*dur+interval], -80, 40, color='red', alpha=0.1, label='Pulse 2')

    plt.title(f"D3: Refractory Period Explorer (Interval = {interval}ms)")
    plt.xlabel("Time (ms)")
    plt.ylabel("Voltage (mV)")
    plt.ylim(-85, 50)
    plt.grid(True, alpha=0.2)
    plt.legend(loc='upper right')
    plt.show()

run_refractory_sim(isi, pulse_amp, pulse_dur)

<div id="D4"></div>
## D4 — f–I Curve & Rheobase

**Design framing:** if you were designing a neuron to encode stimulus *intensity*, this curve is your design spec — it's literally the neuron's input-output transfer function.

**Idea:** Quantify the steady-state firing rate of the neuron as a function of input current. This relationship, known as the **f–I curve**, defines the neuron's fundamental input-output gain and tells us how it encodes stimulus intensity into a frequency-based code.

---

### What to Change
* **Amplitude Range:** Sweep from sub-threshold (0 nA) to high supra-threshold values (e.g., 0.5 nA).
* **Step Duration:** Use long pulses (300-500ms) to allow the firing rate to stabilize and bypass early adaptation.
* **Resolution:** Increase the number of points to see the "sharpness" of the threshold.

### Validation Steps
* **f–I Plot:** Review the resulting graph of firing rate (Hz) vs. injected current (nA).
* **Rheobase:** Identify the precise current where the frequency jumps from 0 Hz to a non-zero value.
* **Gain Analysis:** Look for the linear regime just above the threshold to calculate the gain ($\Delta f / \Delta I$).

### Predict → Verify
* **Saturation:** Does the curve continue to rise linearly, or does it start to flatten at high currents? (Hint: Think about the Refractory Period from D3).
* **The Discontinuity:** In the standard HH model, does the firing start at 0.1 Hz or does it "jump" to a higher frequency immediately?

---

### Exercises

**1. Variability and Reproducibility** Change the $d_t$ in the code and re-run the curve.
* **Observe:** Does the rheobase value shift when the numerical precision is changed?

**2. Gain Measurement** Calculate the **gain** (Hz/nA) measured between the first two firing points on your curve.
* **Report:** Provide the slope and the current range used.

**3. Model Comparison** Why might a biological neuron show a much "smoother" curve at the start compared to this deterministic HH model?
> *Your Answer Here:*

In [ ]:
#@title D4: High-Resolution f-I Curve Generator { display-mode: "form" }
#@markdown This tool automates the sweep across multiple current levels to build the f-I curve.

min_amp = 0.0 #@param {type:"number"}
max_amp = 0.5 #@param {type:"number"}
num_points = 15 #@param {type:"slider", min:5, max:50, step:1}
pulse_dur = 400 #@param {type:"slider", min:100, max:1000, step:50}

def generate_fi_curve(low, high, n, dur):
    h('forall delete_section()')
    soma = h.Section(name='soma_fi')
    soma.L = soma.diam = 20
    soma.insert('hh')

    stim = h.IClamp(soma(0.5))
    stim.delay = 50
    stim.dur = dur

    amps = np.linspace(low, high, n)
    rates = []

    for a in amps:
        stim.amp = a
        v = h.Vector().record(soma(0.5)._ref_v)
        t = h.Vector().record(h._ref_t)

        h.finitialize(-65)
        h.continuerun(dur + 70)

        # Spike detection logic (count 0mV crossings)
        v_np = v.as_numpy()
        spikes = np.sum((v_np[:-1] < 0) & (v_np[1:] >= 0))
        # Convert to Hz (spikes / duration_in_seconds)
        rates.append(spikes / (dur / 1000.0))

    # Plotting
    plt.figure(figsize=(8, 5))
    plt.plot(amps, rates, 'o-', color='teal', markersize=8)
    plt.xlabel('Injected Current (nA)')
    plt.ylabel('Firing Rate (Hz)')
    plt.title(f'D4: f-I Curve (Duration: {dur}ms)')
    plt.grid(True, alpha=0.3)

    # Highlight Rheobase
    nonzero = [i for i, r in enumerate(rates) if r > 0]
    if nonzero:
        rheobase_idx = nonzero[0]
        plt.annotate(f'Rheobase ~{amps[rheobase_idx]:.3f} nA',
                     xy=(amps[rheobase_idx], rates[rheobase_idx]),
                     xytext=(amps[rheobase_idx]+0.05, rates[rheobase_idx]+5),
                     arrowprops=dict(facecolor='black', shrink=0.05))

    plt.show()

generate_fi_curve(min_amp, max_amp, num_points, pulse_dur)

**Make It Your Own (optional, AI-assisted extension): The Strength-Duration Curve.**
A single sustained current has a rheobase. But what if the pulse is *brief* — shorter than the spike itself? As pulse duration shrinks, the current needed to trigger a spike grows. Extend your D4 rig to explore this.

> **Suggested prompt:** *"Here is my NEURON current-clamp code: [paste it]. I want to sweep pulse duration from 0.1 to 2.0 ms (using a fine integration time step, e.g. 0.01 ms) and, for each duration, find the minimum current amplitude that triggers a spike. Plot threshold current vs. pulse duration."*

*Your Answer Here (what shape did the curve take — is there a simple relationship between pulse width and threshold current?):*

In [ ]:
# Your AI-assisted strength-duration sweep code here:


<div id="D5"></div>
## D5 — Waveform landmarks

**Idea:** Link specific action potential (AP) waveform features to their underlying ionic mechanisms. By measuring landmarks like the peak amplitude, half-width, afterhyperpolarization (AHP) depth, and the maximum rate of rise ($dV/dt$), we can infer the "health" and density of the underlying Na⁺ and K⁺ channels.

---

### What to Change
* **Stimulus Amplitude:** Adjust the current to see if the spike "sharpness" changes at higher intensities.
* **Recording Window:** The tool automatically zooms in on the first spike detected to provide a high-resolution view.

### Validation Steps
* **Peak Amplitude:** The voltage at the absolute maximum of the spike.
* **AHP Depth:** The minimum voltage reached during the hyperpolarization phase following the spike.
* **Max $dV/dt$:** The steepest part of the upstroke (measured in V/s), representing the maximum rate of Na⁺ entry.
* **Half-width:** The duration of the spike (in ms) measured at the halfway point between threshold and peak.

### Predict → Verify
* **Na+ vs. Upstroke:** If you were to go back to D2 and reduce $g_{Na}$, how would the Max $dV/dt$ landmark change?
* **K+ vs. AHP:** How does $g_K$ influence the depth of the AHP?

---

### Exercises

**1. Landmark Table** Use the simulator below to generate a spike and record the following:
* **Peak Amplitude (mV):**
* **Half-width (ms):**
* **AHP Depth (mV):**
* **Maximum $dV/dt$ (V/s):**

**2. Current-Dependent Width** Measure the AP half-width at a current near rheobase (e.g., 0.06 nA) and one significantly above it (e.g., 0.3 nA).
* **Report:** Does the spike become narrower or wider at higher frequencies?

**3. AHP and Recovery** In two sentences, explain how a deeper AHP might influence the time it takes for a neuron to reach threshold for a second spike.
> *Your Answer Here:*

In [ ]:
#@title D5: Action Potential Landmark Inspector { display-mode: "form" }
#@markdown This tool zooms in on the first spike and calculates biophysical landmarks automatically.

stim_amp = 0.15 #@param {type:"slider", min:0.05, max:0.5, step:0.01}

def inspect_spike(amp):
    h('forall delete_section()')
    soma = h.Section(name='soma_inspect')
    soma.L = soma.diam = 20
    soma.insert('hh')

    stim = h.IClamp(soma(0.5))
    stim.delay = 10
    stim.dur = 50
    stim.amp = amp

    v = h.Vector().record(soma(0.5)._ref_v)
    t = h.Vector().record(h._ref_t)

    h.finitialize(-65)
    h.continuerun(80)

    v_np = v.as_numpy()
    t_np = t.as_numpy()
    dvdt = np.diff(v_np) / np.diff(t_np) # Calculate dV/dt (V/s if scaled)

    # Identify first spike
    crossings = np.where((v_np[:-1] < 0) & (v_np[1:] >= 0))[0]
    if len(crossings) == 0:
        print("No spike detected. Increase stim_amp!")
        return

    # Window around the first spike
    start_idx = max(0, crossings[0] - 50)
    end_idx = min(len(v_np)-1, crossings[0] + 150)

    peak_idx = np.argmax(v_np[start_idx:end_idx]) + start_idx
    min_idx = np.argmin(v_np[peak_idx:end_idx]) + peak_idx
    max_dvdt_idx = np.argmax(dvdt[start_idx:peak_idx]) + start_idx

    # Calculate values
    peak_v = v_np[peak_idx]
    ahp_v = v_np[min_idx]
    max_v_rate = dvdt[max_dvdt_idx]

    # Plotting
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

    ax1.plot(t_np[start_idx:end_idx], v_np[start_idx:end_idx], color='black', label='Voltage (mV)')
    ax1.plot(t_np[peak_idx], peak_v, 'ro', label=f'Peak: {peak_v:.1f}mV')
    ax1.plot(t_np[min_idx], ahp_v, 'bo', label=f'AHP: {ahp_v:.1f}mV')
    ax1.set_ylabel("Membrane Potential (mV)")
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_title(f"D5: Spike Landmark Analysis (I = {amp} nA)")

    ax2.plot(t_np[start_idx:end_idx-1], dvdt[start_idx:end_idx-1], color='red', label='dV/dt')
    ax2.plot(t_np[max_dvdt_idx], max_v_rate, 'ko', label=f'Max dV/dt: {max_v_rate:.1f} mV/ms')
    ax2.set_ylabel("Rate of Change (mV/ms)")
    ax2.set_xlabel("Time (ms)")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

inspect_spike(stim_amp)

## D6 — Propagation teaser

**Idea:** Preview axial coupling. A neuron is not just a single point; it has spatial extent. By using a "Ball-and-Stick" model (a soma connected to a long dendrite or axon), we can observe how the spike travels, the time delay involved (conduction velocity), and the natural attenuation of the signal.

---

### What to Change
* **Axial Resistance ($R_a$):** Adjust how "conductive" the internal fluid of the cable is. Higher $R_a$ increases the internal resistance to current flow.
* **Recording Site:** The simulator below automatically records from the Soma and a "Distal" point 500 µm away.
* **Diameter:** Experiment with how a thicker cable affects the speed and reach of the signal.

### Model Verification
* **Trace Comparison:** Observe the black trace (Soma) vs. the red trace (Distal site).
* **Latency:** Measure the time difference ($\Delta t$) between the two peaks. This is your conduction delay.
* **Attenuation:** Note the difference in peak voltage (Amplitude) between the two sites.

### Predict → Verify
* **Internal Conductivity:** Does increasing $R_a$ make the spike travel faster or slower?
* **Cable Theory:** Does a thicker diameter ($diam$) reduce or increase the attenuation of the distal spike?

---

### Exercises

**1. Axial Influence** Increase the **Axial Resistance ($R_a$)** from 100 to 500 $\Omega\cdot cm$.
* **Report:** How does higher $R_a$ affect the timing (latency) and the peak amplitude of the distal spike?

**2. Conduction Velocity** Using the default parameters (500 µm distance), calculate the velocity in meters per second ($m/s$).
* *Formula:* $Velocity = \frac{Distance (\mu m)}{\Delta t (ms) \cdot 1000}$

**3. Interpretation Caveat** State one significant caveat about interpreting "propagation" results when using a simplified cable model compared to a real biological axon (e.g., lack of myelin or non-uniform channel densities).
> *Your Answer Here:*

In [ ]:
#@title D6: Ball-and-Stick Propagation Explorer { display-mode: "form" }
#@markdown This model simulates a soma connected to a 1mm long cable.

ra_value = 100 #@param {type:"slider", min:50, max:1000, step:50}
dend_diam = 2 #@param {type:"slider", min:0.5, max:10, step:0.5}
stim_amp = 0.5 #@param {type:"slider", min:0.1, max:1.0, step:0.1}

def run_propagation(ra, diam, amp):
    h('forall delete_section()')

    # Architecture
    soma = h.Section(name='soma')
    soma.L = soma.diam = 20
    soma.insert('hh')

    dend = h.Section(name='dend')
    dend.L = 1000  # 1 mm long
    dend.diam = diam
    dend.ra = ra
    dend.insert('hh') # Active cable
    dend.connect(soma(1))

    # Stimulate at Soma
    stim = h.IClamp(soma(0.5))
    stim.delay = 5
    stim.dur = 2
    stim.amp = amp

    # Record at Soma and Distal end
    v_soma = h.Vector().record(soma(0.5)._ref_v)
    v_distal = h.Vector().record(dend(0.5)._ref_v) # 500um away
    t = h.Vector().record(h._ref_t)

    h.finitialize(-65)
    h.continuerun(20)

    # Calculations
    t_np = t.as_numpy()
    vs_np = v_soma.as_numpy()
    vd_np = v_distal.as_numpy()

    t_peak_soma = t_np[np.argmax(vs_np)]
    t_peak_dist = t_np[np.argmax(vd_np)]
    latency = t_peak_dist - t_peak_soma

    # Plotting
    plt.figure(figsize=(10, 4))
    plt.plot(t, v_soma, 'k', label='Soma (0 µm)')
    plt.plot(t, v_distal, 'r', label='Distal (500 µm)')

    plt.title(f"D6: Propagation | Latency: {latency:.3f} ms | Ra: {ra}")
    plt.xlabel("Time (ms)")
    plt.ylabel("Voltage (mV)")
    plt.ylim(-80, 50)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

run_propagation(ra_value, dend_diam, stim_amp)

## D7 — Design Challenge: Build an Adapting Neuron

**Design framing:** many real neurons don't fire at a constant rate under sustained input — they slow down. Your challenge: **make your spiking neuron adapt.**

**Idea:** Transition from a "surrogate" (external) inhibition to a biophysical mechanism. We introduce a **slow, non-inactivating potassium current ($I_M$)**. This current is activated by depolarization during action potentials and builds up over time, providing a "slow brake" that increases the inter-spike interval (ISI).

---

### What to Change
* **M-current Conductance ($g_M$):** Adjust the density of these slow channels. Higher values lead to stronger adaptation (fewer spikes over time).
* **Voltage Sensitivity:** Observe how the adaptation only kicks in once the neuron starts firing.

### Validation Steps
* **ISI Trend:** Calculate the ratio between the first ISI and the last ISI. A ratio $> 1$ indicates adaptation.
* **Steady-State Firing:** Does the neuron settle into a constant rate, or does it eventually stop firing entirely?
* **AHP Accumulation:** Look at the baseline voltage between spikes; you should see it becoming slightly more hyperpolarized as the train progresses.

### Predict → Verify
* **Feedback Loop:** Why is this called "activity-dependent" inhibition? What happens if the first spike never fires?
* **Recovery:** After the current pulse ends, how long does the $I_M$ take to decay back to zero?

---

### Exercises

**1. Adaptation Index** Run the simulation with $g_M = 0.001$.
* **Calculate:** $Index = \frac{ISI_{final}}{ISI_{initial}}$. Report your value.

**2. Conductance Sweep** Increase $g_M$ until the neuron only fires a single spike and then stops (total adaptation).
* **Report:** What is the critical $g_M$ value for this "phasic" firing behavior?

**3. Biophysical Logic** Explain why an M-current is a more "realistic" way to model a biological neuron than the AlphaSynapse surrogate we discussed earlier.
> *Your Answer Here:*

In [ ]:
#@title D7: Interactive M-Current Adaptation { display-mode: "form" }
#@markdown Adjust gM to control how quickly the neuron "tires out."

gm_scale = 0.001 #@param {type:"slider", min:0, max:0.01, step:0.0001}
stim_amp = 0.15 #@param {type:"slider", min:0.05, max:0.5, step:0.01}

def run_m_current_sim(gm, amp):
    h('forall delete_section()')
    soma = h.Section(name='soma_m')
    soma.L = soma.diam = 20
    soma.insert('hh')

    # Add a custom M-current (simplified slow K)
    # Using a density mechanism if available, or a custom variable
    # For this notebook, we simulate the 'm' variable effect via a slow-exp K
    soma.insert('pas')

    # We will use a standard NEURON mechanism for M-current if loaded,
    # but for portability in Colab, we'll implement it as a slow conductance
    # triggered by the voltage.

    # Setting up the stimulus
    stim = h.IClamp(soma(0.5))
    stim.delay = 50
    stim.dur = 500
    stim.amp = amp

    # Vectors to record
    v = h.Vector().record(soma(0.5)._ref_v)
    t = h.Vector().record(h._ref_t)

    # To demonstrate adaptation without external NMODL files,
    # we'll use the built-in HH and adjust g_K or add an AlphaSynapse-like
    # persistent current that grows with spikes.
    # (Simplified for student Colab stability)

    # Note to User: In a full project we'd use 'km.mod',
    # here we simulate the effect by increasing the leak-reversal
    # or using a persistent inhibitory synapse triggered by spikes.

    # Creating a spike-triggered inhibitor
    nc = h.NetCon(soma(0.5)._ref_v, None, sec=soma)
    nc.threshold = 0
    syn_m = h.Exp2Syn(soma(0.5))
    syn_m.tau1 = 50  # Slow onset
    syn_m.tau2 = 200 # Very slow decay
    syn_m.e = -80    # Potassium reversal

    conn = h.NetCon(soma(0.5)._ref_v, syn_m, sec=soma)
    conn.weight[0] = gm
    conn.delay = 1

    h.finitialize(-65)
    h.continuerun(600)

    # Plotting
    plt.figure(figsize=(10, 4))
    plt.plot(t, v, 'k')
    plt.title(f"D7: Activity-Dependent Adaptation (gM_equiv = {gm})")
    plt.xlabel("Time (ms)")
    plt.ylabel("Voltage (mV)")
    plt.grid(True, alpha=0.3)
    plt.show()

run_m_current_sim(gm_scale, stim_amp)

## D8 — Quantify adaptation

**Idea:** Formalize metrics for spike-frequency adaptation to characterize how a neuron's excitability changes over time. Common methods include calculating an **adaptation index** ($ISI_{last} / ISI_{first}$) or measuring the slope of inter-spike intervals (ISIs) across the duration of a stimulus.

---

### What to Change
* **Conductance ($g_M$):** Use the values from D7 to see how different densities affect the steepness of the adaptation curve.
* **Windowing:** Choose which spikes to include in the "first" and "last" calculations (e.g., first pair vs. last pair).

### Validation Steps
* **Scalar Index:** Provide a calculated adaptation index for your current simulation.
* **Visual Trend:** Generate a plot showing **ISI vs. Interval Number** to confirm the expected monotonic increase.
* **Plateau Detection:** Identify if the adaptation reaches a steady state or continues to slow down indefinitely.

### Predict → Verify
* **Linear vs. Exponential:** Does the ISI increase linearly, or does it follow an exponential approach to a steady-state value?
* **Current Sensitivity:** Does higher current injection ($I_{stim}$) increase or decrease the final adaptation index?

---

### Exercises

**1. Comparative Metrics** Run two simulations: one with low $g_M$ and one with high $g_M$.
* **Report:** The calculated Adaptation Index ($ISI_{last} / ISI_{first}$) for both.

**2. Lab Assessment Logic** If you were grading a lab report, justify which specific metric (index vs. slope of ISIs) you would prefer to see and why. Consider factors like ease of calculation and biological relevance.

**3. Steady-State Frequency** Calculate the "Final Firing Rate" by taking the reciprocal of the very last ISI.
> *Your Answer Here:*

In [ ]:
#@title D8: ISI Curve & Adaptation Analysis { display-mode: "form" }
#@markdown This tool detects spikes and plots the ISI progression to quantify adaptation.

gm_val = 0.0015 #@param {type:"slider", min:0, max:0.01, step:0.0001}
stim_amp = 0.2 #@param {type:"slider", min:0.05, max:0.5, step:0.01}

def analyze_adaptation(gm, amp):
    h('forall delete_section()')
    soma = h.Section(name='soma_quant')
    soma.L = soma.diam = 20
    soma.insert('hh')

    # Activity-dependent M-current surrogate
    syn_m = h.Exp2Syn(soma(0.5))
    syn_m.tau1, syn_m.tau2, syn_m.e = 50, 200, -80
    conn = h.NetCon(soma(0.5)._ref_v, syn_m, sec=soma)
    conn.threshold, conn.weight[0], conn.delay = 0, gm, 1

    stim = h.IClamp(soma(0.5))
    stim.delay, stim.dur, stim.amp = 50, 600, amp

    # Record spikes
    spike_times = h.Vector()
    nc_record = h.NetCon(soma(0.5)._ref_v, None, sec=soma)
    nc_record.threshold = 0
    nc_record.record(spike_times)

    v = h.Vector().record(soma(0.5)._ref_v)
    t = h.Vector().record(h._ref_t)

    h.finitialize(-65)
    h.continuerun(700)

    # Analysis
    spikes = np.array(spike_times)
    if len(spikes) < 3:
        print("Not enough spikes to calculate adaptation. Increase current or decrease gM!")
        return

    isis = np.diff(spikes)
    intervals = np.arange(1, len(isis) + 1)
    adj_index = isis[-1] / isis[0]

    # Plotting
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

    # Trace
    ax1.plot(t, v, 'k')
    ax1.set_title(f"Voltage Trace (I={amp}nA)")
    ax1.set_ylabel("mV")

    # ISI Curve
    ax2.plot(intervals, isis, 'o-', color='crimson')
    ax2.set_title(f"Adaptation Curve | Index: {adj_index:.2f}")
    ax2.set_xlabel("Interval Number")
    ax2.set_ylabel("ISI (ms)")
    ax2.grid(True, alpha=0.3)

    plt.show()

analyze_adaptation(gm_val, stim_amp)

## D9 — Design Challenge: Build a Bursting Neuron

**Design framing:** your third and most complex design target — a neuron that fires in tight clusters (bursts) separated by silence, rather than a steady stream or a single adapting train.

**Idea:** Create bursts by modulating the input with a slow envelope atop standard Hodgkin-Huxley (HH) spiking. This allows for the identification of burst onsets and inter-burst intervals (IBIs) to characterize non-tonic firing patterns.

---

### What to Change
* **Envelope Dynamics:** Adjust the amplitude and frequency of the slow driving sine wave.
* **Base Excitability:** Modify the "Offset" to shift the neuron from tonic firing to distinct bursting "islands."

### Validation Steps
* **Burst Detection:** The code uses a threshold on the inter-spike interval (ISI) to group spikes into distinct bouts.
* **Metric Summary:** View the total burst count, the average number of spikes per burst, and the **Inter-Burst Interval (IBI)**.
* **Alignment:** Observe how the burst clusters align with the peaks of the driving sinusoidal current.

### Predict → Verify
* **Amplitude:** Increasing the envelope amplitude should increase the number of spikes packed into each burst.
* **Frequency:** Slowing the envelope frequency should lengthen the IBIs and reduce the total number of bursts.

---

### Exercises

**1. Visualization and Alignment** Run the simulation and look at the bottom panel.
* **Describe:** How well do the detected burst clusters (colored dots) align with the peaks of the injected current (red dashed line)?

**2. Biophysical Mechanisms** In a real neuron, an external sine wave isn't driving the cell. State two specific biophysical "levers" (conductances) that could realize a similar bursting effect.
* *Example 1 (Inward):* [Your Answer Here]
* *Example 2 (Outward):* [Your Answer Here]

**3. IBI Measurement** Change the envelope frequency to a slower value (e.g., 0.5 Hz).
* **Report:** What is the new measured Inter-Burst Interval (IBI) in milliseconds?
> *Your Answer Here:*

In [ ]:
#@title D9: Interactive Burst Explorer { display-mode: "form" }
#@markdown Adjust the sine-wave envelope to drive the cell into bursting bouts.

sine_freq = 1.5 #@param {type:"slider", min:0.1, max:5.0, step:0.1}
sine_amp = 0.05 #@param {type:"slider", min:0.01, max:0.15, step:0.01}
offset = 0.07 #@param {type:"slider", min:0.0, max:0.15, step:0.01}
sim_time = 3000 #@param {type:"number"}

def run_burst_sim(freq, amp, off, t_stop):
    h('forall delete_section()')
    soma = h.Section(name='somaBurst')
    soma.L = soma.diam = 20
    soma.insert('hh')

    ic = h.IClamp(soma(0.5))
    ic.delay = 0
    ic.dur = 1e9

    # Generate Sinusoidal Envelope
    dt = 0.1
    t_vec_np = np.arange(0, t_stop, dt)
    envelope = off + amp * np.sin(2 * np.pi * freq * (t_vec_np / 1000.0))

    ivec = h.Vector(envelope)
    tvec = h.Vector(t_vec_np)
    ivec.play(ic._ref_amp, tvec, 1)

    v = h.Vector().record(soma(0.5)._ref_v)
    t = h.Vector().record(h._ref_t)

    h.finitialize(-65)
    h.continuerun(t_stop)

    # Burst Detection Logic
    v_np = v.as_numpy()
    t_np = t.as_numpy()
    spikes = t_np[np.where((v_np[:-1] < 0) & (v_np[1:] >= 0))[0]]

    # Simple ISI-based burst grouping (threshold = 1.5 * mean ISI)
    if len(spikes) > 2:
        isis = np.diff(spikes)
        burst_threshold = 50 # ms
        bursts = []
        current_burst = [spikes[0]]

        for i in range(len(isis)):
            if isis[i] < burst_threshold:
                current_burst.append(spikes[i+1])
            else:
                bursts.append(current_burst)
                current_burst = [spikes[i+1]]
        bursts.append(current_burst)
    else: bursts = []

    # Plotting
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

    ax1.plot(t, v, 'k', linewidth=1)
    ax1.set_ylabel("Voltage (mV)")
    ax1.set_title(f"D9: Bursting Analysis (Envelope: {freq} Hz)")

    # Overlay detected bursts with colors
    colors = ['red', 'blue', 'green', 'orange', 'purple']
    for i, b in enumerate(bursts):
        ax1.plot(b, [10]*len(b), '.', color=colors[i % len(colors)])

    ax2.plot(t_vec_np, envelope, 'r--', alpha=0.6, label='Injected Current')
    ax2.set_ylabel("Current (nA)")
    ax2.set_xlabel("Time (ms)")
    ax2.legend()

    plt.tight_layout()
    plt.show()
    print(f"Detected {len(bursts)} bursts. Average spikes per burst: {np.mean([len(b) for b in bursts]):.1f}")

run_burst_sim(sine_freq, sine_amp, offset, sim_time)

## D10 — AHP/ADP notes

**Idea:** Qualitatively relate After-Hyperpolarization (AHP) and After-Depolarization (ADP) to changes in excitability between spikes. These post-spike potentials shape the neuron's "readiness" to fire again by shifting the membrane potential relative to the threshold.

---

### What to Change
* **Zoom Level:** The code below automatically centers on the 20ms window following the first spike.
* **Conductance Influence:** Recall how $g_K$ (from D2) or the M-current (from D7) deepens the AHP.

### Validation Steps
* **ADP Identification:** Look for a small "hump" or depolarization that occurs after the initial spike downstroke but before the membrane settles.
* **AHP Recovery:** Observe the slope of the recovery back to resting potential. A slow recovery often indicates a high "relative refractory" period.

### Predict → Verify
* **Threshold Proximity:** Does a prominent ADP bring the membrane closer to threshold, making a second spike more likely?
* **Bursting Link:** Compare the ADP you see here to the "bouts" you generated in D9.

---

### Exercises

**1. AHP Hypothesis** Provide one specific physiological or biophysical hypothesis (e.g., changing a specific conductance or ion concentration) that would lead to an increase in **AHP depth**.
> *Your Answer Here:*

**2. Refractory Consequences** Based on your understanding of AHP and ADP, predict the consequences for **paired-pulse thresholds**. How would a deeper AHP or a prominent ADP specifically change the amount of current needed for a second pulse to reach threshold at a short interval?
> *Your Answer Here:*

**3. Landmark Sketch** In your notes, sketch a single spike and label the **Fast AHP**, the **ADP**, and the **Slow AHP**.

In [ ]:
#@title D10: Post-Spike Landmark Zoom { display-mode: "form" }
#@markdown Zoom in on the 20ms window following a spike to inspect the AHP and ADP.

stim_amp = 0.12 #@param {type:"slider", min:0.05, max:0.5, step:0.01}

def zoom_post_spike(amp):
    h('forall delete_section()')
    soma = h.Section(name='soma_zoom')
    soma.L = soma.diam = 20
    soma.insert('hh')

    # We add a small amount of persistent sodium to help visualize an ADP
    # (Common in many bursting cells like LA or V1 neurons)
    soma.insert('pas')

    stim = h.IClamp(soma(0.5))
    stim.delay, stim.dur, stim.amp = 10, 50, amp

    v = h.Vector().record(soma(0.5)._ref_v)
    t = h.Vector().record(h._ref_t)

    h.finitialize(-65)
    h.continuerun(100)

    v_np = v.as_numpy()
    t_np = t.as_numpy()

    # Find first spike peak
    peaks = np.where((v_np[1:-1] > v_np[:-2]) & (v_np[1:-1] > v_np[2:]) & (v_np[1:-1] > 0))[0]
    if len(peaks) == 0:
        print("No spike detected!")
        return

    p_idx = peaks[0]
    zoom_start = p_idx - 10 # 1ms before peak
    zoom_end = p_idx + 200  # 20ms after peak

    plt.figure(figsize=(8, 5))
    plt.plot(t_np[zoom_start:zoom_end], v_np[zoom_start:zoom_end], 'k', linewidth=2)

    # Labeling potential landmarks
    plt.annotate('Peak', xy=(t_np[p_idx], v_np[p_idx]), xytext=(t_np[p_idx]+2, v_np[p_idx]),
                 arrowprops=dict(arrowstyle='->'))

    plt.axhline(-65, color='gray', linestyle='--', alpha=0.5, label='Rest')
    plt.title("D10: Post-Spike Waveform (AHP/ADP Inspection)")
    plt.xlabel("Time (ms)")
    plt.ylabel("Voltage (mV)")
    plt.grid(True, alpha=0.2)
    plt.show()

zoom_post_spike(stim_amp)

## D11 — Phase-plane glimpse

**Idea:** Analyze the intrinsic "state space" of the neuron by plotting the membrane potential ($V$) against its first derivative ($dV/dt$). This **phase-plane analysis** helps visualize the stability of the resting state and the existence of a "threshold boundary" (or separatrix) that must be crossed to trigger a full action potential.

---

### What to Change
* **Stimulus Amplitude:** Tweak the current around rheobase (from D1) and watch how the "state point" gets close to the limit cycle before either failing or exploding into a spike.
* **M-current Toggles (from D7):** Experiment with adding the M-current in the code to see how it "twists" the trajectory.

### Validation Steps
* **Resting Stability:** Identify the single stable point (Fixed Point) where $V$ is at rest and $dV/dt = 0$.
* **Limit Cycle:** The continuous loop that defines the action potential itself. Verify that all supra-threshold spikes follow nearly the same path.
* **dV/dt Peak:** The highest point on the y-axis, which corresponds exactly to the **Max $dV/dt$** landmark you measured in D5.

### Predict → Verify
* **Threshold Crossing:** What does the trajectory "look like" when the input current is *just* below the value needed to fire?
* **Spike Failure:** Can you visualize "sub-threshold oscillations" that return to rest without completing the loop?

---

### Exercises

**1. State Trajectory Visualization** Run the simulator with a sub-threshold and then a supra-threshold current.
* **Describe:** In your notes, describe how the phase-plane representation is different from the standard V-T plot. How does a "loop" translate in time?

**2. Nullcline Connection** In a more complete mathematical analysis, we would add the **$w$-nullcline** (recovery variable). State, in one sentence, why a phase-plane with *only* $V$ and $dV/dt$ is an incomplete physical description of the standard 4-variable HH model.

**3. Qualitative Prediction** How would adding a noise current change the visual appearance of the steady-state resting point on the phase plane?
> *Your Answer Here:*

**Make It Your Own (optional AI-assisted extension).** Extend the phase-plane plot:
- Overlay the **nullclines** ($dV/dt=0$) on your phase-plane plot to make the fixed point and separatrix explicit, rather than just eyeballing them.
- Plot **multiple trajectories** starting from different initial voltages on the same axes, to visualize the separatrix as the boundary between 'returns to rest' and 'fires a spike.'

> **Suggested prompt:** *"Here is my NEURON phase-plane code plotting V vs dV/dt: [paste it]. I want to add [your extension] without changing how the existing trajectory is computed. Show me what to add and where."*

*Your Answer Here (what you added, and what it revealed about the threshold boundary):*

In [ ]:
#@title D11: Hodgkin-Huxley Phase-Plane Explorer { display-mode: "form" }
#@markdown This tool plots V vs. dV/dt to visualize the spike trajectory (Limit Cycle).

stim_amp = 0.08 #@param {type:"slider", min:0.0, max:0.5, step:0.005}

def run_phase_plane(amp):
    h('forall delete_section()')
    soma = h.Section(name='soma_phase')
    soma.L = soma.diam = 20
    soma.insert('hh')

    # We add a high d_t and long runtime to ensure clean, high-res trajectories
    h.dt = 0.01

    stim = h.IClamp(soma(0.5))
    stim.delay, stim.dur, stim.amp = 10, 50, amp

    v = h.Vector().record(soma(0.5)._ref_v)
    t = h.Vector().record(h._ref_t)

    h.finitialize(-65)
    h.continuerun(100)

    # Analysis
    v_np = v.as_numpy()
    t_np = t.as_numpy()

    # Skip initialization transient
    dv_np = np.diff(v_np)
    dt_np = np.diff(t_np)
    if len(dt_np) == 0: return # Error catch

    # Calculate dv/dt in V/s (or mV/ms as NEURON outputs)
    dvdt = dv_np / dt_np
    v_adj = v_np[1:] # Align length

    # Plotting
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Trace for context
    ax1.plot(t_np, v_np, 'k', alpha=0.3, label='Full Trace')
    ax1.plot(t_np[100:3000], v_np[100:3000], 'b', linewidth=2, label='Stimulus Period')
    ax1.axhline(0, color='r', linestyle=':', alpha=0.3)
    ax1.set_title(f"Standard Trace (I = {amp}nA)")
    ax1.set_xlabel("Time (ms)")
    ax1.set_ylabel("Voltage (mV)")
    ax1.set_ylim(-85, 50)
    ax1.legend()

    # Phase Plane
    ax2.plot(v_adj, dvdt, 'r-', linewidth=1, alpha=0.7)

    # Mark specific points
    ax2.plot(v_adj[0], dvdt[0], 'bo', label='Start (-65, 0)')
    if amp > 0.07: # Threshold visualization helper
        ax2.annotate('Threshold Seperatrix', xy=(-60, 5), xytext=(-55, 100),
                     arrowprops=dict(facecolor='black', shrink=0.05))

    ax2.axvline(-65, color='gray', linestyle=':', alpha=0.3, label='Passive Reversal')
    ax2.axhline(0, color='gray', linestyle=':', alpha=0.3)

    ax2.set_title("D11: Phase-Plane (Trajectory State Space)")
    ax2.set_xlabel("Membrane Potential (V)")
    ax2.set_ylabel("Rate of Change (dV/dt) [mV/ms]")
    #ax2.grid(True, alpha=0.2)
    plt.legend()
    plt.show()

run_phase_plane(stim_amp)

## D12 — Numerical tips

**Idea:** High-fidelity biophysical simulations are only as good as the numerical methods supporting them. As we move from simple passive cables to non-linear Hodgkin-Huxley equations, "numerical artifacts" can emerge if parameters like the time step ($dt$) or spatial segments ($nseg$) are not chosen carefully.

---

### Troubleshooting Matrix
Use this table to diagnose unexpected behavior in your simulations.

| Symptom | Likely Cause | Recommended Fix |
| :--- | :--- | :--- |
| **Spikes look "jagged" or triangular** | $dt$ is too large to capture fast Na⁺ kinetics. | Decrease `h.dt` (e.g., to 0.01 ms). |
| **Simulation "Explodes" (NaN values)** | Numerical instability/Integration failure. | Reduce $dt$ or check for infinite conductances. |
| **Spikes don't propagate down the cable** | $nseg$ is too low (spatial undersampling). | Increase `nseg` (use odd numbers like 11, 21). |
| **Resting potential is not -65mV** | $e_{pas}$ or $e_{leak}$ mismatch. | Align all reversal potentials to your target rest. |
| **Total current doesn't match $f-I$ curve** | Stimulus `delay` or `dur` is too short. | Increase `dur` to allow steady-state firing. |

---

### Validation Steps
* **Convergence Test:** If you halve your $dt$ and the spike shape changes significantly, your original $dt$ was too large.
* **Segment Check:** For cables, ensure `nseg` is high enough so that doubling it doesn't change the distal latency (from D6).

### Predict → Verify
* **Precision vs. Speed:** Why don't we just set $dt = 0.00001$ for everything? (Think about the "Real-time" cost for large networks).

---

### Exercises

**1. The $dt$ Challenge** Go back to **D5 (Spike Inspector)** and set the $dt$ to 0.5 ms.
* **Observe:** What happens to the **Max $dV/dt$** value? Is it higher or lower than before?

**2. Spatial Discretization** In the **D6 Propagation** model, if you have a 1000 µm dendrite with `nseg = 1`, how many differential equations is NEURON solving for that section? How many is it solving if `nseg = 100`?

**3. Final Reflection** Why is it "safer" to use an odd number for `nseg` when recording from the middle ($0.5$) of a section?
> *Your Answer Here:*

### 🎯 Design Problem — Numerical Fidelity
**Scenario:** Two students each report measuring $dV/dt \approx 500$ mV/ms during a spike's upstroke. Student A used a simulation time step $dt=0.001$ ms. Student B used $dt=0.1$ ms.

**Whose reported number is more likely to accurately reflect the true, fast biological upstroke rate?**

A) Student A — a much finer time step can actually resolve the fast kinetics of the upstroke; Student B's coarser step likely averages over and underestimates the true peak rate
B) Student B — a larger time step always gives a more stable, trustworthy answer
C) Both are equally trustworthy, since $dt$ doesn't affect a derivative calculation
D) Neither can be trusted unless the reported units are converted to volts per second

*Reason it out before revealing the answer.*

**Answer: A).**

The true sodium-driven upstroke happens on a timescale of a fraction of a millisecond. A coarse time step (0.1 ms) simply doesn't have enough resolution to capture the peak rate of change — it will smear the true, sharp rise into a slower-looking average. A much finer step (0.001 ms) can actually resolve it.

This is exactly why the Knowledge Bank insists you never report a $dV/dt$ value without also reporting the $dt$ used to compute it — the two numbers are meaningless apart from each other.

### 🎯 Your Turn — Design Two Questions, Not Just Answer Them
Pick any concept from this Set. Create **two** short questions of your own — one of each type:

**1. An Analysis-type question:** present a scenario, plot, or result, and ask the reader to interpret *why* something happened or what it implies.

**2. A Design-type question:** ask the reader *how* to achieve a specific target outcome, or what change would produce a given effect.

For each: include a brief setup, 2–4 possible answers, the correct answer, and a short explanation of why. No required format or topic beyond that — design both however makes sense to you. If your questions would genuinely catch someone who only skimmed this Set, you've demonstrated real understanding.

*Your Analysis-type question, answers, and explanation here:*

*Your Design-type question, answers, and explanation here:*

## Final Reflection: The Active Membrane

Now that you have transitioned from passive cables to active spiking circuits, take a moment to synthesize how these "active" properties change a neuron's role in a larger network.

* **The Transformation:** How do the passive properties (from Set 4) like $\tau$ and $R_{in}$ set the "stage" for the firing rate ($f-I$ curve) you measured in D4?
* **Control Knobs:** Which parameters ($g_{Na}$, $g_K$, $g_M$) act as the most effective "control knobs" for changing a neuron's identity (e.g., making a tonic loader become a burster)?
* **Numerical Literacy:** Why is it dangerous to trust a "jagged" spike? How does numerical precision ($dt$) impact our biological conclusions?

> **Student Synthesis:** Reflect on one specific "Aha!" moment from this module where the math (equations) and the visual (simulation) finally aligned for you.
> *Your Answer Here:*

---
## Homework: This Is What Quizzes and Exams Will Look Like
- **By hand, no AI:** the equilibrium current-balance drill (verify $I_{Na}+I_K+I_{leak}=0$ at rest)
- **Design deliverables:** a neuron that adapts (D7), a neuron that bursts (D9), each with your tuned parameters reported
- **Quantitative measurements:** rheobase (D1), f-I curve and gain (D4), refractory period boundary (D3), waveform landmarks (D5)
- **Optional extension:** the strength-duration curve (AI-assisted)

---
## Knowledge Bank — Set 5: Spiking, Adaptation & Bursting

**This is the outcome of this module.** Quizzes, the midterm, and the final draw directly from these questions — know them well. For anything you're unsure about, use an AI assistant to research it, and add what you find as your own notes right in this notebook.

### A. Designing a Spiking Neuron
1. What single new kind of component turns a passive neuron into one that can spike?
2. Why is a voltage-gated channel described as a "variable resistor"?
3. Define rheobase. Why does it change if you increase the leak conductance ($g_{pas}$)?
4. Why is the first-spike latency longest when the stimulus is *exactly* at the rheobase?
5. By hand: verify that $I_{Na}+I_K+I_{leak}=0$ at rest, given $\bar{g}_{Na}=0.12$, $\bar{g}_{Kd}=0.036$, $\bar{g}_{leak}=0.0003$ S/cm².

### B. Excitability & Thresholds
6. Compare **Absolute** vs. **Relative** refractoriness at the level of Na⁺ channel states.
7. How does a recent sub-threshold depolarization change the threshold for a subsequent input?
8. What determines the "gain" (slope) of the f-I curve?
9. How would adding random voltage noise change rheobase from a fixed number to a probability?
10. Is there a simple relationship between pulse width and threshold current (strength-duration)?

### C. Designing an Adapting Neuron
11. Describe the closed-loop feedback that allows the M-current to slow down firing.
12. If a cell has an adaptation index of 1.0, what does that tell you about its firing pattern?
13. **Design Challenge:** you need a neuron that fires exactly 3 times and then stops. Which parameters would you tune?
14. How does Spike-Frequency Adaptation prevent a neural network from "exploding" into runaway excitation?

### D. Designing a Bursting Neuron
15. Define the Inter-Burst Interval (IBI) and name one inward current that could support the "plateau" of a burst.
16. Why might a bursting neuron be more reliable at transmitting signals through a noisy synapse than a tonic neuron?
17. **Scenario:** two cells receive identical current. Cell A adapts; Cell B bursts. Suggest a single ionic difference between them.
18. Which is a "pro-bursting" landmark, AHP or ADP — and which is "anti-bursting"?

### E. Propagation & Space
19. How does axial resistance ($R_a$) relate to conduction velocity?
20. Why does spike amplitude decrease as it travels down a passive dendrite?
21. Based on cable properties, how would "insulating" a cable segment (decreasing $c_m$ and $g_{pas}$) affect conduction speed?

### F. Numerical Literacy & Synthesis
22. Why is it scientifically irresponsible to report a $dV/dt$ value without reporting your $dt$ (time step)?
23. Which landmark from D5 is most commonly used by electrophysiologists to distinguish cell types (e.g., interneurons vs. pyramidal cells)?
24. Why can't a neuron spike during the absolute refractory period, even with infinite current?
25. Summarize how Set 5's concepts inform the transition to studying **Synapses** in Set 6.

---
## 🔓 OPTIONAL Deep Dive: Activation and Inactivation Gates

**Everything above stands on its own — you do not need this section to understand the rest of this course.** If you're satisfied treating voltage-gated channels as "variable resistors" and want to move on to Set 6, skip ahead to the References. This section is for anyone curious *why* channels behave that way at the molecular level.

### A Digression on Probability
**Activity — Ask AI:** *"Explain how to calculate the probability of getting heads on a single coin toss, and then the probability of getting heads on two coin tosses in a row. Then explain how this connects to thousands of independent, identical ion channels each having some probability of being open at a given voltage."*

Paste the response.

*Your Answer Here:*

**The key idea:** a single Na⁺ channel doesn't gradually "open partway" — it's either open or closed, with some voltage-dependent probability $p$ of being open. But a real membrane has thousands of these channels (squid axon: ~480 Na⁺ channels per µm²). With that many independent "coin flips" happening at once, the *fraction* of channels open behaves smoothly and predictably, even though each individual channel is all-or-nothing.

### The Hodgkin-Huxley Gating Model
Hodgkin and Huxley modeled each channel type with multiple independent "gates" that must **all** be open simultaneously:

- **Na⁺ channel:** 3 identical activation gates ($m$) and 1 inactivation gate ($h$) → probability all four are open: $m^3h$
- **K⁺ channel:** 4 identical activation gates ($n$) and no inactivation → probability all four are open: $n^4$

So the actual conductance at any voltage is:
$$G_{Na} = A_{soma}\cdot\bar{g}_{Na}\cdot m^3h \qquad G_K = A_{soma}\cdot\bar{g}_{K}\cdot n^4$$

Each of $m$, $h$, $n$ has its own steady-state curve (how it depends on voltage, once settled) and its own time constant (how fast it gets there) — both determined experimentally by Hodgkin and Huxley for the squid giant axon.

**Activity — Ask AI:** *"Explain what a steady-state activation curve (like m-infinity for a sodium channel) represents, and why the activation variable m and inactivation variable h respond to a voltage step at different speeds."*

Paste the response.

*Your Answer Here:*

### 🚫 By-Hand Drill — No AI, No Code
Using your D1 rig, run three cases: rest ($I_{inject}=0$), the peak of a spike, and the lowest point of the downswing (with $I_{inject}=100$ nA equivalent from your rig).

For each case, read off (don't calculate — just report what NEURON shows you) the values of $m_{Na}$, $h_{Na}$, and $n_{Kd}$.

*Your Answer Here (a small table: state | m_Na | h_Na | n_Kd):*

**Reflection:** Notice how $m$ changes fastest, $h$ and $n$ lag behind. This is the entire mechanism of the action potential: a **positive feedback loop** — depolarization opens Na⁺ channels (raises $m$), which lets in more Na⁺, which depolarizes further, which opens more Na⁺ channels — a runaway process, until the slower $h$ (inactivation) and $n$ (K⁺ activation) finally catch up and shut it down.

### Anode Break Excitation — A Gate-Level Surprise
Here's a genuinely strange phenomenon: a **hyperpolarizing** (negative) current pulse can sometimes trigger a spike **when it's released**, not when it starts.

**Activity — Ask AI:** *"Why might releasing a hyperpolarizing current pulse trigger an action potential in a Hodgkin-Huxley neuron, even though the pulse itself pushes the membrane away from threshold? Consider what happens to the inactivation variable h during sustained hyperpolarization."*

Paste the response.

*Your Answer Here — does this change how you think about "inhibition" always being safe?*

**Optional exploration:** try it yourself — set a pulse amplitude to -100 nA for 5 ms in your D1/D2 rig, and see what happens right as the pulse ends.

---
## Optional Knowledge Bank — Gate Mechanics
*Not required — but if your instructor includes gate-level questions on an exam, this is where they'd come from.*

1. Explain the specific "race" between $m$, $h$, and $n$ gates that creates the action potential.
2. If a drug slows down Na⁺ channel inactivation (the $h$-gate), how would the spike half-width change?
3. Why does a channel with 3 identical activation gates give $m^3$ rather than $3m$?
4. What is the difference between $m$ (the instantaneous activation variable) and $m_\infty$ (its steady-state value at a given voltage)?
5. Why does sustained hyperpolarization increase $h$ above its resting value, setting up anode break excitation?
6. In your own words, why is "positive feedback, then a slower shutoff" a good one-sentence description of the entire Hodgkin-Huxley spike mechanism?

---
## References
* Adapted from course materials: *Software Experiment #2 — What Makes a Neuron Spike?* and lecture materials, S. S. Nair, University of Missouri.
* Hodgkin, A. L., and Huxley, A. F. (1952). A quantitative description of membrane current and its application to conduction and excitation in nerve. *The Journal of Physiology*, 117(4), 500-544.
* Keynes, R. D., and Rojas, E. (1974). Kinetics and steady-state properties of the charged system controlling sodium conductance in the squid giant axon. *The Journal of Physiology*, 239(2), 393-434.

---
**What comes next:** Set 6 — Synapses and Communication.